In [12]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image, ImageFilter
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULT_DIR = Path('./results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ── Detection parameters ──────────────────────────────────────────────────────
PEAK_MIN_DIST   = 5      # NMS minimum distance (px)
PEAK_THRESH     = 35     # Peak intensity threshold (35=all, 60=bright only, 80=brightest)
TOP_N           = 15     # Maximum number of spots to retain (brightest first)
RADIUS_MAX_KEEP = 8.0    # Discard spots with radius > this value (px); None to disable
RADIUS_FWHM     = 0.5    # FWHM fraction for radius estimation
RADIUS_MIN      = 2.0    # Minimum spot radius (px)
RADIUS_MAX      = 18.0   # Maximum spot radius (px)
MASK_DILATE_PX  = 15     # Nuclear mask dilation (px)
SMOOTH_R        = 1.5    # Pre-detection Gaussian blur radius (PIL)
NMS_WIN         = 7      # Local-maximum window (px)

# ── Inpainting parameters ─────────────────────────────────────────────────────
ANNULAR_FACTOR  = 2.5    # Outer annulus radius = spot_radius × ANNULAR_FACTOR
FEATHER_RATIO   = 0.30   # Feather width ratio
MIN_ANNULAR_W   = 4      # Minimum annulus width (px)

# ── Physical scale ────────────────────────────────────────────────────────────
PIX_NM  = 147.6          # Pixel size (nm/px)
_PIX_NM2 = PIX_NM ** 2  # nm²/px²

# ── Batch / group-comparison mode ─────────────────────────────────────────────

PARENT_DIR = Path('./data')  # group-comparison mode

BATCH_DIR  = None              
BATCH_EXTS = {'.tif'}

# IMAGE_PATH: auto-set to first image in first sub-folder of PARENT_DIR
_subdirs = sorted([d for d in PARENT_DIR.iterdir() if d.is_dir()]) if (PARENT_DIR and PARENT_DIR.is_dir()) else []
_imgs    = sorted([p for p in _subdirs[0].iterdir() if p.suffix.lower() in BATCH_EXTS]) if _subdirs else []
IMAGE_PATH = _imgs[0] if _imgs else None


In [13]:
def gauss_fft(arr, sigma):
    """FFT-based Gaussian filter."""
    H, W = arr.shape
    A    = np.fft.fft2(arr.astype(np.float64))
    fy, fx = np.fft.fftfreq(H), np.fft.fftfreq(W)
    FX, FY = np.meshgrid(fx, fy)
    G = np.exp(-2 * np.pi**2 * sigma**2 * (FX**2 + FY**2))
    return np.real(np.fft.ifft2(A * G))


def fill_holes_np(mask):
    """BFS-based hole filling for nuclear mask."""
    from numpy import pad
    padded  = pad(mask, 1, constant_values=0)
    visited = np.zeros_like(padded, dtype=bool)
    stack   = [(0, 0)]
    while stack:
        r, c = stack.pop()
        if r < 0 or r >= padded.shape[0] or c < 0 or c >= padded.shape[1]:
            continue
        if visited[r, c] or padded[r, c]:
            continue
        visited[r, c] = True
        stack += [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]
    return mask | ~visited[1:-1, 1:-1]


def estimate_radius(img, cy, cx, peak_val, fwhm_frac=0.5, r_min=2.0, r_max=18.0):
    """Estimate spot radius as the median distance at which intensity
    drops below peak_val * fwhm_frac along 16 radial profiles."""
    H, W   = img.shape
    angles = np.linspace(0, 2*np.pi, 16, endpoint=False)
    target = peak_val * fwhm_frac
    radii  = []
    for a in angles:
        for r in np.arange(1, r_max + 1, 0.5):
            ry = int(round(cy + r * np.sin(a)))
            rx = int(round(cx + r * np.cos(a)))
            if ry < 0 or ry >= H or rx < 0 or rx >= W:
                radii.append(r); break
            if img[ry, rx] < target:
                radii.append(r); break
        else:
            radii.append(r_max)
    return float(np.clip(np.median(radii), r_min, r_max))


In [14]:
# Load image
img_raw = np.array(Image.open(IMAGE_PATH))
img_f   = img_raw.astype(np.float64)
H, W    = img_f.shape

# Nuclear mask (threshold + hole filling + dilation)
nuc_sm         = np.array(Image.fromarray(img_raw).filter(ImageFilter.GaussianBlur(radius=3)))
nuc_mask       = fill_holes_np(nuc_sm > 5)
nuc_mask_dilated = np.array(
    Image.fromarray(nuc_mask.astype(np.uint8) * 255)
    .filter(ImageFilter.MaxFilter(size=MASK_DILATE_PX * 2 + 1))
) > 0


In [15]:
# Gaussian smoothing
img_smooth = np.array(
    Image.fromarray(img_raw).filter(ImageFilter.GaussianBlur(radius=SMOOTH_R))
).astype(np.float64)

# Local-maximum detection via MaxFilter approximation
img_sm_u8  = np.clip(img_smooth, 0, 255).astype(np.uint8)
img_maxflt = np.array(Image.fromarray(img_sm_u8).filter(ImageFilter.MaxFilter(size=NMS_WIN)))
local_max  = (img_sm_u8 == img_maxflt) & (img_sm_u8 >= PEAK_THRESH) & nuc_mask_dilated
peaks_yx   = list(zip(*np.where(local_max)))


In [16]:
# NMS: keep brightest peak within PEAK_MIN_DIST
peaks_yx = sorted(peaks_yx, key=lambda p: img_raw[p[0], p[1]], reverse=True)
kept = []
for (py, px) in peaks_yx:
    if not any((py-ky)**2 + (px-kx)**2 < PEAK_MIN_DIST**2 for ky, kx in kept):
        kept.append((py, px))
kept = kept[:TOP_N]


In [17]:
yy, xx = np.mgrid[0:H, 0:W]
records    = []
spot_union = np.zeros((H, W), dtype=bool)

for idx, (by, bx) in enumerate(kept):
    peak_val = float(img_smooth[by, bx])
    r = estimate_radius(img_smooth, by, bx, peak_val,
                        fwhm_frac=RADIUS_FWHM, r_min=RADIUS_MIN, r_max=RADIUS_MAX)
    if RADIUS_MAX_KEEP is not None and r > RADIUS_MAX_KEEP:
        continue
    bm = ((yy - by)**2 + (xx - bx)**2) <= r**2
    bm &= nuc_mask_dilated
    if bm.sum() == 0:
        continue
    spot_union |= bm
    pix = img_raw[bm].astype(float)
    records.append(dict(
        spot_id            = len(records) + 1,
        cy                 = float(by),
        cx                 = float(bx),
        radius_px          = round(r, 2),
        area_px2           = int(bm.sum()),
        area_nm2           = round(int(bm.sum()) * _PIX_NM2, 1),
        peak_intensity     = int(img_raw[by, bx]),
        mean_intensity     = round(pix.mean(), 2),
        max_intensity      = round(pix.max(), 2),
        integrated_density = round(pix.sum(), 1),
    ))

df      = pd.DataFrame(records)
n       = len(df)
nuc_px  = img_raw[nuc_mask].astype(float)
pix_in  = img_raw[spot_union].astype(float)
pix_out = img_raw[nuc_mask & ~spot_union].astype(float)
df.to_csv(RESULT_DIR / 'spot_metrics.csv', index=False)


In [18]:
plt.rcParams.update({
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'figure.facecolor': '#f5f5f5'
})
BCOL, NCOL = '#e63946', '#457b9d'

fig, axes = plt.subplots(2, 4, figsize=(16, 9), dpi=130)
fig.subplots_adjust(hspace=0.45, wspace=0.42, left=0.06, right=0.97, top=0.91, bottom=0.08)

# A: Original image
ax = axes[0, 0]
ax.imshow(img_raw, cmap='gray'); ax.set_title('Original image', fontweight='bold'); ax.axis('off')

# B: Detected Spots
ax = axes[0, 1]
ax.imshow(img_raw, cmap='gray')
ax.contour(nuc_mask.astype(float), levels=[0.5], colors=['#90e0ef'], linewidths=0.8)
ax.contour(nuc_mask_dilated.astype(float), levels=[0.5], colors=['#f4a261'], linewidths=0.6, linestyles='--')
for _, row in df.iterrows():
    ax.add_patch(mpatches.Circle((row['cx'], row['cy']), row['radius_px'],
                                 fill=False, edgecolor=BCOL, lw=1.5))
    ax.text(row['cx'], row['cy'], str(int(row['spot_id'])),
            color='yellow', fontsize=7, ha='center', va='center', fontweight='bold')
ax.set_title(f'Detected Spots (n={n})\ncyan=mask  orange=dilated', fontweight='bold'); ax.axis('off')

# C: Spot radius (bar chart)
ax = axes[0, 2]
if n > 0:
    df_s = df.sort_values('radius_px', ascending=False).reset_index(drop=True)
    ax.bar(range(n), df_s['radius_px'], color=NCOL, edgecolor='white', lw=0.7)
    ax.set_xticks(range(n))
    ax.set_xticklabels([f'B{int(r.spot_id)}' for _, r in df_s.iterrows()], fontsize=8, rotation=45)
    ax.axhline(df['radius_px'].mean(), color='red', lw=1.4, ls='--',
               label=f'mean={df["radius_px"].mean():.1f} px')
    ax.legend(fontsize=8)
ax.set_ylabel('Radius (px)'); ax.set_title('Spot radius', fontweight='bold')

# D: Peak intensity (bar chart)
ax = axes[0, 3]
if n > 0:
    df_s2 = df.sort_values('peak_intensity', ascending=False).reset_index(drop=True)
    ax.bar(range(n), df_s2['peak_intensity'], color=BCOL, edgecolor='white', lw=0.7)
    ax.set_xticks(range(n))
    ax.set_xticklabels([f'B{int(r.spot_id)}' for _, r in df_s2.iterrows()], fontsize=8, rotation=45)
    ax.axhline(df['peak_intensity'].mean(), color='navy', lw=1.4, ls='--',
               label=f'mean={df["peak_intensity"].mean():.1f}')
    ax.legend(fontsize=8)
ax.set_ylabel('Peak intensity (a.u.)'); ax.set_title('Spot peak intensity', fontweight='bold')

# E: Intensity histogram (spot vs. non-spot)
ax = axes[1, 0]
bins = np.arange(0, int(nuc_px.max()) + 5, 2)
ax.hist(pix_out, bins=bins, color=NCOL, alpha=0.75, label=f'Non-Spot (n={len(pix_out):,})')
if len(pix_in):
    ax.hist(pix_in, bins=bins, color=BCOL, alpha=0.80, label=f'Dense Spots (n={len(pix_in):,})')
ax.set_xlabel('Pixel intensity (a.u.)'); ax.set_ylabel('Pixel count')
ax.set_title('Nucleus intensity distribution\n(spot vs. diffuse)', fontweight='bold')
ax.legend(fontsize=8)

# F: Smoothed intensity distribution
ax = axes[1, 1]
def smooth_hist(data, bins, k=5):
    counts, edges = np.histogram(data, bins=bins, density=True)
    sm = np.convolve(counts, np.ones(k)/k, mode='same')
    return (edges[:-1]+edges[1:])/2, sm
x_lin = np.linspace(0, nuc_px.max(), 100)
if len(nuc_px) > 1:
    c, s = smooth_hist(nuc_px, bins=x_lin)
    ax.fill_between(c, s, alpha=0.35, color='gray', label='All nucleus'); ax.plot(c, s, color='gray', lw=1)
if len(pix_in) > 1:
    c2, s2 = smooth_hist(pix_in, bins=x_lin)
    ax.fill_between(c2, s2, alpha=0.55, color=BCOL, label='Dense Spots'); ax.plot(c2, s2, color=BCOL, lw=1.2)
ax.set_xlabel('Pixel intensity (a.u.)'); ax.set_ylabel('Density')
ax.set_title('Intensity distribution\n(nucleus vs. spots)', fontweight='bold'); ax.legend(fontsize=8)

# G: Radius vs. peak intensity
ax = axes[1, 2]
if n > 0:
    sc = ax.scatter(df['radius_px'], df['peak_intensity'],
                    c=df['integrated_density'], cmap='YlOrRd',
                    s=100, edgecolors='k', lw=0.7, zorder=3)
    plt.colorbar(sc, ax=ax, label='Integrated density', shrink=0.85)
    for _, row in df.iterrows():
        ax.annotate(f"B{int(row['spot_id'])}", (row['radius_px'], row['peak_intensity']),
                    fontsize=7.5, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Radius (px)'); ax.set_ylabel('Peak intensity (a.u.)')
ax.set_title('Size vs. peak intensity\n(color = integrated density)', fontweight='bold')

# H: Summary text
ax = axes[1, 3]; ax.axis('off')
lines = [
    f"{'─'*28}", f" {IMAGE_PATH.name}", f" {W}×{H} px",
    f"{'─'*28}", " NUCLEUS",
    f"  Area  : {nuc_mask.sum():,} px²",
    f"  Mean  : {nuc_px.mean():.1f}", f"  Std   : {nuc_px.std():.1f}",
    f"  Max   : {int(nuc_px.max())}",
    f"{'─'*28}", f" BLOBS (n={n})",
]
if n > 0:
    lines += [
        f"  R     : {df['radius_px'].mean():.1f}±{df['radius_px'].std():.1f} px",
        f"  R rng : {df['radius_px'].min():.1f}–{df['radius_px'].max():.1f}",
        f"  Peak  : {df['peak_intensity'].mean():.1f} mean",
        f"  Area% : {100*spot_union.sum()/nuc_mask.sum():.1f}%",
    ]
lines.append(f"{'─'*28}")
ax.text(0.03, 0.97, '\n'.join(lines), transform=ax.transAxes,
        va='top', ha='left', fontsize=8.5, fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#aaaaaa', lw=1))

fig.suptitle('DAPI/Hoechst Nuclear DNA Dense-Spot Analysis  |  Local-max detection (PIL+NumPy)',
             fontsize=12, fontweight='bold', y=0.97)
out = RESULT_DIR / 'spot_analysis_panel.png'
fig.savefig(str(out), dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close(fig)
print(f'Saved: {out}')


Saved: results\spot_analysis_panel.png


In [19]:
img_inpaint = img_raw.astype(np.float64).copy()

for _, row in df.iterrows():
    by, bx, r = row['cy'], row['cx'], row['radius_px']
    outer_r   = max(r * ANNULAR_FACTOR, r + MIN_ANNULAR_W)
    dist      = np.sqrt((yy - by)**2 + (xx - bx)**2)

    am = (dist > r) & (dist <= outer_r) & nuc_mask_dilated
    if am.sum() == 0:
        am = (dist > r) & (dist <= outer_r * 2) & nuc_mask_dilated
    bg_val = float(img_raw[am].mean()) if am.sum() > 0 else float(img_raw[nuc_mask].mean())

    fw     = max(r * FEATHER_RATIO, 1.0)
    weight = np.where(dist <= (r - fw), 1.0, np.clip((r - dist) / fw, 0.0, 1.0))
    weight[dist > r] = 0.0
    img_inpaint = img_inpaint * (1.0 - weight) + bg_val * weight

img_inpaint_u8 = np.clip(img_inpaint, 0, 255).astype(np.uint8)
img_inpaint_u8 = np.array(Image.fromarray(img_inpaint_u8).filter(ImageFilter.GaussianBlur(radius=1)))
Image.fromarray(img_inpaint_u8).save(str(RESULT_DIR / 'spot_inpainted.tif'))

# Comparison figure
BCOL = '#e63946'
fig2, axes2 = plt.subplots(1, 3, figsize=(13, 4.5), dpi=130)
fig2.patch.set_facecolor('#f5f5f5')
axes2[0].imshow(img_raw, cmap='gray', vmin=0, vmax=img_raw.max())
for _, row in df.iterrows():
    axes2[0].add_patch(mpatches.Circle((row['cx'], row['cy']), row['radius_px'],
                                       fill=False, edgecolor=BCOL, lw=1.5))
    axes2[0].text(row['cx'], row['cy'], str(int(row['spot_id'])),
                  color='yellow', fontsize=7, ha='center', va='center', fontweight='bold')
axes2[0].set_title('Original + spots', fontweight='bold'); axes2[0].axis('off')
axes2[1].imshow(img_inpaint_u8, cmap='gray', vmin=0, vmax=img_raw.max())
axes2[1].set_title('Spot-inpainted', fontweight='bold'); axes2[1].axis('off')
diff   = img_raw.astype(np.int16) - img_inpaint_u8.astype(np.int16)
vmax_d = max(abs(diff).max(), 1)
im = axes2[2].imshow(diff, cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
axes2[2].set_title(f'Difference (orig − inpainted)  max Δ={vmax_d}', fontweight='bold'); axes2[2].axis('off')
plt.colorbar(im, ax=axes2[2], shrink=0.85, label='Intensity diff (a.u.)')
fig2.suptitle('Spot Inpainting — local annular background fill', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
cmp_path = RESULT_DIR / 'spot_inpaint_comparison.png'
fig2.savefig(str(cmp_path), dpi=130, bbox_inches='tight', facecolor=fig2.get_facecolor())
plt.close(fig2)
print(f'Saved: {cmp_path}')


Saved: results\spot_inpaint_comparison.png


In [20]:
def analyze_one(img_path):
    """Analyse a single image. Returns (df, img_raw, img_inpaint_u8,
    nuc_mask, nuc_mask_dilated, spot_union, img_smooth)."""
    img_raw = np.array(Image.open(img_path))
    H, W    = img_raw.shape
    yy, xx  = np.mgrid[0:H, 0:W]

    # Nuclear mask
    nuc_sm         = np.array(Image.fromarray(img_raw).filter(ImageFilter.GaussianBlur(radius=3)))
    nuc_mask       = fill_holes_np(nuc_sm > 5)
    nuc_mask_dilated = np.array(
        Image.fromarray(nuc_mask.astype(np.uint8) * 255)
        .filter(ImageFilter.MaxFilter(size=MASK_DILATE_PX * 2 + 1))
    ) > 0

    # Smoothing
    img_smooth = np.array(
        Image.fromarray(img_raw).filter(ImageFilter.GaussianBlur(radius=SMOOTH_R))
    ).astype(np.float64)

    # Local-maximum detection
    img_sm_u8  = np.clip(img_smooth, 0, 255).astype(np.uint8)
    img_maxflt = np.array(Image.fromarray(img_sm_u8).filter(ImageFilter.MaxFilter(size=NMS_WIN)))
    local_max  = (img_sm_u8 == img_maxflt) & (img_sm_u8 >= PEAK_THRESH) & nuc_mask_dilated
    peaks_yx   = sorted(list(zip(*np.where(local_max))),
                        key=lambda p: img_raw[p[0], p[1]], reverse=True)

    # NMS
    kept = []
    for (py, px) in peaks_yx:
        if not any((py-ky)**2 + (px-kx)**2 < PEAK_MIN_DIST**2 for ky, kx in kept):
            kept.append((py, px))
    kept = kept[:TOP_N]

    # Measurement
    records, spot_union = [], np.zeros((H, W), dtype=bool)
    for (by, bx) in kept:
        peak_val = float(img_smooth[by, bx])
        r = estimate_radius(img_smooth, by, bx, peak_val,
                            fwhm_frac=RADIUS_FWHM, r_min=RADIUS_MIN, r_max=RADIUS_MAX)
        if RADIUS_MAX_KEEP is not None and r > RADIUS_MAX_KEEP:
            continue
        bm = ((yy - by)**2 + (xx - bx)**2) <= r**2
        bm &= nuc_mask_dilated
        if bm.sum() == 0:
            continue
        spot_union |= bm
        pix = img_raw[bm].astype(float)
        records.append(dict(
            spot_id=len(records)+1, cy=float(by), cx=float(bx),
            radius_px=round(r, 2), area_px2=int(bm.sum()),
            area_nm2=round(int(bm.sum()) * _PIX_NM2, 1),
            peak_intensity=int(img_raw[by, bx]),
            mean_intensity=round(pix.mean(), 2),
            max_intensity=round(pix.max(), 2),
            integrated_density=round(pix.sum(), 1),
            polar_r=round(float(np.sqrt((bx - W/2)**2 + (by - H/2)**2)), 2),
            polar_theta_deg=round(float(np.degrees(np.arctan2(by - H/2, bx - W/2))), 2),
        ))

    df = pd.DataFrame(records)

    # Inpainting
    img_inpaint = img_raw.astype(np.float64).copy()
    for _, row in df.iterrows():
        by2, bx2, r2 = row['cy'], row['cx'], row['radius_px']
        outer_r = max(r2 * ANNULAR_FACTOR, r2 + MIN_ANNULAR_W)
        dist    = np.sqrt((yy - by2)**2 + (xx - bx2)**2)
        am = (dist > r2) & (dist <= outer_r) & nuc_mask_dilated
        if am.sum() == 0:
            am = (dist > r2) & (dist <= outer_r * 2) & nuc_mask_dilated
        bg_val = float(img_raw[am].mean()) if am.sum() > 0 else float(img_raw[nuc_mask].mean())
        fw = max(r2 * FEATHER_RATIO, 1.0)
        w  = np.where(dist <= (r2 - fw), 1.0, np.clip((r2 - dist) / fw, 0.0, 1.0))
        w[dist > r2] = 0.0
        img_inpaint = img_inpaint * (1.0 - w) + bg_val * w
    img_inpaint_u8 = np.clip(img_inpaint, 0, 255).astype(np.uint8)
    img_inpaint_u8 = np.array(
        Image.fromarray(img_inpaint_u8).filter(ImageFilter.GaussianBlur(radius=1))
    )
    return df, img_raw, img_inpaint_u8, nuc_mask, nuc_mask_dilated, spot_union, img_smooth


def save_outputs(stem, df, img_raw, img_inpaint_u8,
                 nuc_mask, nuc_mask_dilated, spot_union, result_dir, img_path):
    """Save panel.png, _spots.csv, _inpainted.tif, _inpaint_comparison.png."""
    Path(result_dir).mkdir(parents=True, exist_ok=True)
    H, W    = img_raw.shape
    n       = len(df)
    nuc_px  = img_raw[nuc_mask].astype(float)
    pix_in  = img_raw[spot_union].astype(float) if spot_union.any() else np.array([])
    pix_out = img_raw[nuc_mask & ~spot_union].astype(float)
    BCOL, NCOL = '#e63946', '#457b9d'

    df.to_csv(Path(result_dir) / f'{stem}_spots.csv', index=False)

    plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                         'axes.grid': True, 'grid.alpha': 0.3, 'figure.facecolor': '#f5f5f5'})
    fig, axes = plt.subplots(2, 4, figsize=(16, 9), dpi=130)
    fig.subplots_adjust(hspace=0.45, wspace=0.42, left=0.06, right=0.97, top=0.91, bottom=0.08)

    axes[0,0].imshow(img_raw, cmap='gray'); axes[0,0].set_title('Original image', fontweight='bold'); axes[0,0].axis('off')

    axes[0,1].imshow(img_raw, cmap='gray')
    axes[0,1].contour(nuc_mask.astype(float), [0.5], colors=['#90e0ef'], linewidths=0.8)
    axes[0,1].contour(nuc_mask_dilated.astype(float), [0.5], colors=['#f4a261'], linewidths=0.6, linestyles='--')
    for _, row in df.iterrows():
        axes[0,1].add_patch(mpatches.Circle((row['cx'],row['cy']),row['radius_px'],fill=False,edgecolor=BCOL,lw=1.5))
        axes[0,1].text(row['cx'],row['cy'],str(int(row['spot_id'])),color='yellow',fontsize=7,ha='center',va='center',fontweight='bold')
    axes[0,1].set_title(f'Detected Spots (n={n})', fontweight='bold'); axes[0,1].axis('off')

    if n > 0:
        df_s = df.sort_values('radius_px', ascending=False).reset_index(drop=True)
        axes[0,2].bar(range(n), df_s['radius_px'], color=NCOL, edgecolor='white', lw=0.7)
        axes[0,2].set_xticks(range(n)); axes[0,2].set_xticklabels([f'B{int(r.spot_id)}' for _,r in df_s.iterrows()], fontsize=8, rotation=45)
        axes[0,2].axhline(df['radius_px'].mean(), color='red', lw=1.4, ls='--', label=f'mean={df["radius_px"].mean():.1f} px'); axes[0,2].legend(fontsize=8)
    axes[0,2].set_ylabel('Radius (px)'); axes[0,2].set_title('Spot radius', fontweight='bold')

    if n > 0:
        df_s2 = df.sort_values('peak_intensity', ascending=False).reset_index(drop=True)
        axes[0,3].bar(range(n), df_s2['peak_intensity'], color=BCOL, edgecolor='white', lw=0.7)
        axes[0,3].set_xticks(range(n)); axes[0,3].set_xticklabels([f'B{int(r.spot_id)}' for _,r in df_s2.iterrows()], fontsize=8, rotation=45)
        axes[0,3].axhline(df['peak_intensity'].mean(), color='navy', lw=1.4, ls='--', label=f'mean={df["peak_intensity"].mean():.1f}'); axes[0,3].legend(fontsize=8)
    axes[0,3].set_ylabel('Peak intensity (a.u.)'); axes[0,3].set_title('Spot peak intensity', fontweight='bold')

    bins = np.arange(0, int(nuc_px.max())+5, 2)
    axes[1,0].hist(pix_out, bins=bins, color=NCOL, alpha=0.75, label=f'Non-Spot (n={len(pix_out):,})')
    if len(pix_in): axes[1,0].hist(pix_in, bins=bins, color=BCOL, alpha=0.80, label=f'Dense Spots (n={len(pix_in):,})')
    axes[1,0].set_xlabel('Pixel intensity (a.u.)'); axes[1,0].set_ylabel('Pixel count')
    axes[1,0].set_title('Nucleus intensity distribution', fontweight='bold'); axes[1,0].legend(fontsize=8)

    def smooth_hist(data, bins, k=5):
        counts, edges = np.histogram(data, bins=bins, density=True)
        sm = np.convolve(counts, np.ones(k)/k, mode='same')
        return (edges[:-1]+edges[1:])/2, sm
    x_lin = np.linspace(0, nuc_px.max(), 100)
    if len(nuc_px) > 1:
        c, s = smooth_hist(nuc_px, bins=x_lin); axes[1,1].fill_between(c, s, alpha=0.35, color='gray', label='All nucleus'); axes[1,1].plot(c, s, color='gray', lw=1)
    if len(pix_in) > 1:
        c2, s2 = smooth_hist(pix_in, bins=x_lin); axes[1,1].fill_between(c2, s2, alpha=0.55, color=BCOL, label='Dense Spots'); axes[1,1].plot(c2, s2, color=BCOL, lw=1.2)
    axes[1,1].set_xlabel('Pixel intensity (a.u.)'); axes[1,1].set_ylabel('Density')
    axes[1,1].set_title('Intensity distribution', fontweight='bold'); axes[1,1].legend(fontsize=8)

    if n > 0:
        sc = axes[1,2].scatter(df['radius_px'], df['peak_intensity'], c=df['integrated_density'],
                               cmap='YlOrRd', s=100, edgecolors='k', lw=0.7, zorder=3)
        plt.colorbar(sc, ax=axes[1,2], label='Integrated density', shrink=0.85)
        for _, row in df.iterrows():
            axes[1,2].annotate(f'B{int(row["spot_id"])}', (row['radius_px'], row['peak_intensity']),
                               fontsize=7.5, xytext=(3,3), textcoords='offset points')
    axes[1,2].set_xlabel('Radius (px)'); axes[1,2].set_ylabel('Peak intensity (a.u.)')
    axes[1,2].set_title('Size vs. peak intensity', fontweight='bold')

    axes[1,3].axis('off')
    lines = [f"{'─'*28}", f' {Path(img_path).name}', f' {W}x{H} px',
             f"{'─'*28}", ' NUCLEUS',
             f'  Area  : {nuc_mask.sum():,} px²', f'  Mean  : {nuc_px.mean():.1f}',
             f'  Std   : {nuc_px.std():.1f}', f'  Max   : {int(nuc_px.max())}',
             f"{'─'*28}", f' BLOBS (n={n})']
    if n > 0:
        lines += [f'  R     : {df["radius_px"].mean():.1f}±{df["radius_px"].std():.1f} px',
                  f'  R rng : {df["radius_px"].min():.1f}–{df["radius_px"].max():.1f}',
                  f'  Peak  : {df["peak_intensity"].mean():.1f} mean',
                  f'  Area% : {100*spot_union.sum()/nuc_mask.sum():.1f}%']
    lines.append(f"{'─'*28}")
    axes[1,3].text(0.03, 0.97, '\n'.join(lines), transform=axes[1,3].transAxes,
                   va='top', ha='left', fontsize=8.5, fontfamily='monospace',
                   bbox=dict(boxstyle='round,pad=0.5', fc='white', ec='#aaaaaa', lw=1))
    fig.suptitle(f'DAPI/Hoechst Nuclear DNA Dense-Spot Analysis  |  {Path(img_path).name}',
                 fontsize=12, fontweight='bold', y=0.97)
    panel_path = Path(result_dir) / f'{stem}_panel.png'
    fig.savefig(str(panel_path), dpi=130, bbox_inches='tight', facecolor=fig.get_facecolor()); plt.close(fig)

    Image.fromarray(img_inpaint_u8).save(str(Path(result_dir) / f'{stem}_inpainted.tif'))

    fig2, axes2 = plt.subplots(1, 3, figsize=(13, 4.5), dpi=130); fig2.patch.set_facecolor('#f5f5f5')
    axes2[0].imshow(img_raw, cmap='gray', vmin=0, vmax=img_raw.max())
    for _, row in df.iterrows():
        axes2[0].add_patch(mpatches.Circle((row['cx'],row['cy']),row['radius_px'],fill=False,edgecolor=BCOL,lw=1.5))
        axes2[0].text(row['cx'],row['cy'],str(int(row['spot_id'])),color='yellow',fontsize=7,ha='center',va='center',fontweight='bold')
    axes2[0].set_title('Original + spots', fontweight='bold'); axes2[0].axis('off')
    axes2[1].imshow(img_inpaint_u8, cmap='gray', vmin=0, vmax=img_raw.max()); axes2[1].set_title('Spot-inpainted', fontweight='bold'); axes2[1].axis('off')
    diff = img_raw.astype(np.int16) - img_inpaint_u8.astype(np.int16); vmax_d = max(abs(diff).max(), 1)
    im = axes2[2].imshow(diff, cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d)
    axes2[2].set_title(f'Difference  max Δ={vmax_d}', fontweight='bold'); axes2[2].axis('off')
    plt.colorbar(im, ax=axes2[2], shrink=0.85, label='Intensity diff (a.u.)')
    fig2.suptitle(f'Spot Inpainting  |  {Path(img_path).name}', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig2.savefig(str(Path(result_dir) / f'{stem}_inpaint_comparison.png'),
                 dpi=130, bbox_inches='tight', facecolor=fig2.get_facecolor()); plt.close(fig2)


In [21]:
_NM2_TO_UM2 = 1e-6
_PX_TO_UM   = PIX_NM / 1000.0

_SW_METRICS = [
    ('spot_count',           'Spot Count',         'count / image', 1.0         ),
    ('mean_area_nm2',        'Mean Area',           'µm²',           _NM2_TO_UM2 ),
    ('sd_area_nm2',          'SD of Area',          'µm²',           _NM2_TO_UM2 ),
    ('mean_mean_intensity',  'Mean Spot Intensity', 'a.u.',          1.0         ),
    ('mean_polar_r',         'Mean Polar r',        'µm',            _PX_TO_UM   ),
    ('mean_polar_theta_deg', 'Mean Polar θ',        'deg',           1.0         ),
]


def _run_batch(target_dir, result_dir):
    """Analyse all images in target_dir; return (summary_df, per_img_df)."""
    files = sorted([p for p in Path(target_dir).iterdir() if p.suffix.lower() in BATCH_EXTS])
    all_dfs = []
    for img_path in files:
        try:
            df_b, img_raw_b, img_ip_b, nuc_b, nuc_dil_b, spot_u_b, img_sm_b = analyze_one(img_path)
            save_outputs(img_path.stem, df_b, img_raw_b, img_ip_b,
                         nuc_b, nuc_dil_b, spot_u_b, result_dir, img_path)
            if len(df_b) > 0:
                df_tagged = df_b.copy(); df_tagged.insert(0, 'image', img_path.name)
                all_dfs.append(df_tagged)
        except Exception as e:
            print(f'[ERROR] {img_path.name}: {e}')
    if not all_dfs:
        return None, None
    summary_df = pd.concat(all_dfs, ignore_index=True)
    records = []
    for img_name, sub in summary_df.groupby('image', sort=False):
        records.append(dict(
            image=img_name, spot_count=len(sub),
            mean_area_nm2=round(sub['area_nm2'].mean(), 1),
            sd_area_nm2=round(float(sub['area_nm2'].std(ddof=0)), 1),
            mean_mean_intensity=round(sub['mean_intensity'].mean(), 3),
            mean_polar_r=round(sub['polar_r'].mean(), 3),
            mean_polar_theta_deg=round(sub['polar_theta_deg'].mean(), 3),
        ))
    return summary_df, pd.DataFrame(records)


def _draw_swarm(axs, per_img_df, metrics, rng, bw=0.25):
    for si, (col, title, ylabel, fac) in enumerate(metrics):
        ax = axs[si]
        v  = per_img_df[col].values.astype(float) * fac
        j  = rng.uniform(-0.15, 0.15, len(v))
        ax.scatter(j, v, s=40, color='black', alpha=0.75, zorder=3, linewidths=0)
        if len(v) > 1:
            m, s = v.mean(), v.std(ddof=0)
            ax.vlines(0, m-s, m+s, colors='red', lw=1.5, zorder=4)
            ax.hlines([m-s, m+s], -bw*0.7, bw*0.7, colors='red', lw=1.5, zorder=4)
            ax.hlines(m, -bw, bw, colors='red', lw=3.0, zorder=5)
        ax.set_xlim(-0.5, 0.5); ax.set_xticks([])
        ax.set_ylabel(ylabel, fontsize=9); ax.set_title(title, fontweight='bold', fontsize=10)
        ax.grid(True, alpha=0.25, axis='y')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.spines['bottom'].set_visible(False)


def _save_batch_results(summary_df, per_img_df, result_dir, label=''):
    result_dir = Path(result_dir); result_dir.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(result_dir / 'batch_summary.csv', index=False)
    try:
        with pd.ExcelWriter(str(result_dir / 'batch_analysis.xlsx'), engine='openpyxl') as w:
            summary_df.to_excel(w, sheet_name='all_spots', index=False)
            per_img_df.to_excel(w, sheet_name='per_image', index=False)
    except Exception:
        pass
    nc, nr = 3, (len(_SW_METRICS) + 2) // 3
    fig_sw, _ = plt.subplots(nr, nc, figsize=(5*nc, 4*nr), dpi=130)
    fig_sw.patch.set_facecolor('white')
    axs_sw = np.array(fig_sw.axes)
    _draw_swarm(axs_sw, per_img_df, _SW_METRICS, np.random.default_rng(42))
    for sj in range(len(_SW_METRICS), len(axs_sw)): axs_sw[sj].set_visible(False)
    ttl = f'Spot Statistics — {label}' if label else 'Spot Statistics'
    fig_sw.suptitle(ttl, fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig_sw.savefig(str(result_dir / 'batch_swarm.png'), dpi=130, bbox_inches='tight',
                   facecolor=fig_sw.get_facecolor()); plt.close(fig_sw)
    print(f'Saved: {result_dir / "batch_swarm.png"}')


# ── Execution ─────────────────────────────────────────────────────────────────
if PARENT_DIR is not None and Path(PARENT_DIR).is_dir():
    subdirs = sorted([d for d in Path(PARENT_DIR).iterdir() if d.is_dir()])
    group_per_img = {}
    for subdir in subdirs:
        gname   = subdir.name
        gresult = Path(RESULT_DIR) / gname
        s_df, p_df = _run_batch(subdir, gresult)
        if s_df is None:
            continue
        _save_batch_results(s_df, p_df, gresult, label=gname)
        group_per_img[gname] = p_df
        print(f'[{gname}] {len(p_df)} images, {len(s_df)} spots total')

    # Group-comparison figure
    if len(group_per_img) >= 2:
        groups  = list(group_per_img.keys())
        COLORS  = ['#e63946','#2196F3','#4caf50','#ff9800','#9c27b0','#00bcd4']
        BW      = 0.20
        nc, nr  = 3, (len(_SW_METRICS) + 2) // 3
        fig_cmp, _ = plt.subplots(nr, nc, figsize=(5*nc, 4*nr), dpi=130)
        fig_cmp.patch.set_facecolor('white')
        axs_cmp = np.array(fig_cmp.axes)
        rng     = np.random.default_rng(42)
        for mi, (col, title, ylabel, fac) in enumerate(_SW_METRICS):
            ax = axs_cmp[mi]
            for gi, gname in enumerate(groups):
                gdf  = group_per_img[gname]
                if col not in gdf.columns: continue
                vals = gdf[col].values.astype(float) * fac
                xc   = float(gi)
                j2   = rng.uniform(-0.15, 0.15, len(vals))
                cc   = COLORS[gi % len(COLORS)]
                ax.scatter(xc + j2, vals, s=45, color=cc, alpha=0.75, zorder=3, linewidths=0)
                if len(vals) >= 1:
                    m, s = vals.mean(), vals.std(ddof=0)
                    ax.vlines(xc, m-s, m+s, colors='black', lw=1.5, zorder=4)
                    ax.hlines([m-s, m+s], xc-BW*0.7, xc+BW*0.7, colors='black', lw=1.5, zorder=4)
                    ax.hlines(m, xc-BW, xc+BW, colors='black', lw=3.0, zorder=5)
            ax.set_xticks(range(len(groups)))
            ax.set_xticklabels(groups, fontsize=9, rotation=15, ha='right')
            ax.set_xlim(-0.6, len(groups) - 0.4)
            ax.set_ylabel(ylabel, fontsize=9); ax.set_title(title, fontweight='bold', fontsize=10)
            ax.grid(True, alpha=0.25, axis='y')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        for sj in range(len(_SW_METRICS), len(axs_cmp)): axs_cmp[sj].set_visible(False)
        fig_cmp.suptitle('Group Comparison — Spot Statistics', fontsize=13, fontweight='bold')
        plt.tight_layout()
        cmp_path = Path(RESULT_DIR) / 'group_comparison.png'
        fig_cmp.savefig(str(cmp_path), dpi=130, bbox_inches='tight', facecolor=fig_cmp.get_facecolor())
        plt.close(fig_cmp)
        print(f'Saved: {cmp_path}')

elif BATCH_DIR is not None and Path(BATCH_DIR).is_dir():
    s_df, p_df = _run_batch(BATCH_DIR, RESULT_DIR)
    if s_df is not None:
        _save_batch_results(s_df, p_df, RESULT_DIR)
        display(p_df)

else:
    (df_s, img_raw_s, img_ip_s,
     nuc_s, nuc_dil_s, spot_u_s, img_sm_s) = analyze_one(IMAGE_PATH)
    save_outputs(IMAGE_PATH.stem, df_s, img_raw_s, img_ip_s,
                 nuc_s, nuc_dil_s, spot_u_s, RESULT_DIR, IMAGE_PATH)
    df_s.to_csv(RESULT_DIR / 'spot_metrics.csv', index=False)
    display(df_s)


Saved: results\Astrocyte\batch_swarm.png
[Astrocyte] 6 images, 65 spots total
